# Passive Voice Analysis by Gender and Topic

Analyses the rate of passive sentences per gender for each topic file in `corpus_normalization/normalised_mp_sentences_by_topic/`.

In [1]:
from PassivePySrc import PassivePy
import pandas as pd
from pathlib import Path

in_dir = Path("../../corpus_normalization/normalised_mp_sentences_by_topic")
out_dir = Path("results_by_topic")
out_dir.mkdir(parents=True, exist_ok=True)

passivepy = PassivePy.PassivePyAnalyzer(spacy_model="en_core_web_lg")

/Users/ameliemajor/miniforge3/envs/passivepy/lib/python3.11/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_core_web_lg' (3.4.0) was trained with spaCy v3.4.0 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
topic_files = sorted(in_dir.glob("*.csv"))
if not topic_files:
    raise FileNotFoundError(f"No topic CSVs found in: {in_dir.resolve()}")

print(f"Found {len(topic_files)} topic files:")
for f in topic_files:
    print(f"  {f.name}")

Found 10 topic files:
  0_covid_virus_vaccine_pandemic.csv
  1_climate_emissions_carbon_energy.csv
  2_scottish_scotland_snp_sturgeon.csv
  3_biden_trump_sanders_democratic.csv
  4_schools_school_education_teachers.csv
  5_farage_ukip_nuttall_party.csv
  6_her_she_may_brexit.csv
  7_book_books_novel_my.csv
  8_nhs_health_doctors_patients.csv
  9_eu_barnier_uk_deal.csv


In [3]:
all_results = []

for file in topic_files:
    topic = file.stem
    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print(f"{'='*60}")

    df = pd.read_csv(file)

    if "sentence" not in df.columns or "gender" not in df.columns:
        print(f"  Skipping — missing 'sentence' or 'gender' column")
        continue

    if len(df) == 0:
        print("  Skipping — empty file")
        continue

    # Run PassivePy
    df_passive = passivepy.match_corpus_level(
        df,
        column_name="sentence",
        n_process=1,
        batch_size=100,
        add_other_columns=True,
        truncated_passive=False,
        full_passive=False,
    )

    # Summarise by gender
    summary = (
        df_passive
        .groupby("gender", dropna=False)
        .agg(
            total_sentences=("passive_count", "size"),
            total_passives=("passive_count", "sum"),
        )
    )
    summary["passive_rate"] = summary["total_passives"] / summary["total_sentences"]
    print(summary)

    male_rate = float(summary.loc["M", "passive_rate"]) if "M" in summary.index else 0.0
    female_rate = float(summary.loc["F", "passive_rate"]) if "F" in summary.index else 0.0

    all_results.append({
        "topic": topic,
        "male_passive_rate": male_rate,
        "female_passive_rate": female_rate,
        "male_total": int(summary.loc["M", "total_sentences"]) if "M" in summary.index else 0,
        "female_total": int(summary.loc["F", "total_sentences"]) if "F" in summary.index else 0,
        "male_passives": int(summary.loc["M", "total_passives"]) if "M" in summary.index else 0,
        "female_passives": int(summary.loc["F", "total_passives"]) if "F" in summary.index else 0,
    })


Topic: 0_covid_virus_vaccine_pandemic
Detecting Sentences...
Starting to find passives...
        total_sentences total_passives passive_rate
gender                                             
F                   111             51     0.459459
M                   676            322     0.476331

Topic: 1_climate_emissions_carbon_energy
Detecting Sentences...
Starting to find passives...
        total_sentences total_passives passive_rate
gender                                             
F                   118             41     0.347458
M                   652            247     0.378834

Topic: 2_scottish_scotland_snp_sturgeon
Detecting Sentences...
Starting to find passives...
        total_sentences total_passives passive_rate
gender                                             
F                   194            103     0.530928
M                  1281            586     0.457455

Topic: 3_biden_trump_sanders_democratic
Detecting Sentences...
Starting to find passives...
     

In [4]:
results_df = pd.DataFrame(all_results)
print("\nAll topics — passive rate by gender:\n")
results_df


All topics — passive rate by gender:



,topic,male_passive_rate,female_passive_rate,male_total,female_total,male_passives,female_passives
0,0_covid_virus_vaccine_pandemic,0.476331,0.459459,676,111,322,51
1,1_climate_emissions_carbon_energy,0.378834,0.347458,652,118,247,41
2,2_scottish_scotland_snp_sturgeon,0.457455,0.530928,1281,194,586,103
3,3_biden_trump_sanders_democratic,0.190476,0.444444,42,9,8,4
4,4_schools_school_education_teachers,0.412008,0.484043,483,188,199,91
5,5_farage_ukip_nuttall_party,0.351351,0.344828,740,145,260,50
6,6_her_she_may_brexit,0.382080,0.451178,971,297,371,134
7,7_book_books_novel_my,0.223881,0.166667,67,6,15,1
8,8_nhs_health_doctors_patients,0.384158,0.446281,505,121,194,54
9,9_eu_barnier_uk_deal,0.422939,0.396396,558,111,236,44


In [5]:
results_path = out_dir / "passive_rates_by_topic.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved results to {results_path}")

Saved results to results_by_topic/passive_rates_by_topic.csv
